In [1]:
!pip install pandas numpy scikit-learn openpyxl gradio flask

   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.4 MB ? eta -:--:--
    --------------------------------------- 0.5/31.4 MB 786.3 kB/s eta 0:00:40
    --------------------------------------- 0.5/31.4 MB 786.3 kB/s eta 0:00:40
   - -------------------------------------- 0.8/31.4 MB 737.0 kB/s eta 0:00:42
   - -------------------------------------- 0.8/31.4 MB 737.0 kB/s eta 0:00:42
   - -------------------------------------- 1.0/31.4 MB 792.9 kB/s eta 0:00:39
   - -------------------------------------- 1.3/31.4 MB 874.1 kB/s eta 0:00:35
   -- ------------------------------------- 1.6/31.4 MB 923.0 kB/s eta 0:00:33
   -- ------------------------------------- 1.8/31.4 MB 975.5 kB/s eta 0:00:31
   --- -----------

In [1]:
import pandas as pd
heart=pd.read_csv(r"D:\ok\Documents\heart_disease.csv")
loan=pd.read_csv(r"D:\ok\Documents\loan_approval.csv")

In [2]:
heart.head()

,age,cholesterol,blood_pressure,smoker,heart_risk
0,33,224,106,1,0
1,66,263,144,1,1
2,61,261,103,0,0
3,50,230,150,1,1
4,50,295,148,0,1


In [3]:
loan.head()

,income,credit_score,loan_amount,employed,loan_approved
0,114,547,10,1,1
1,63,472,36,1,0
2,115,717,24,0,1
3,136,770,36,0,1
4,126,837,37,0,1


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
CONFIG = {
    "heart": {"df": heart, "target": "heart_risk",
              "labels": {0: "Low Risk", 1: "High Risk"}},
    "loan":  {"df": loan, "target": "loan_approved",
              "labels": {0: "Rejected", 1: "Approved"}},
}

MODELS, FEATURES = {}, {}

def train(name):
    cfg = CONFIG[name]
    df = cfg["df"]
    X = df.drop(columns=cfg["target"])
    y = df[cfg["target"]]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression())
    ])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"===== {name.upper()} DATASET =====")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

    MODELS[name] = model
    FEATURES[name] = list(X.columns)

train("heart")
train("loan")

# Used later by the Gradio and Flask interfaces
def predict_proba(name, values):
    X = pd.DataFrame([values], columns=FEATURES[name])
    return MODELS[name].predict_proba(X)[0]   # [P(class 0), P(class 1)]

# Quick test
print(predict_proba("heart", [55, 240, 140, 1]))
print(predict_proba("loan", [70, 700, 20, 1]))

===== HEART DATASET =====
Accuracy: 0.9167
Confusion Matrix:
 [[ 6  1]
 [ 1 16]]
Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.86      0.86         7
           1       0.94      0.94      0.94        17

    accuracy                           0.92        24
   macro avg       0.90      0.90      0.90        24
weighted avg       0.92      0.92      0.92        24

===== LOAN DATASET =====
Accuracy: 0.8333
Confusion Matrix:
 [[ 9  2]
 [ 2 11]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.82      0.82        11
           1       0.85      0.85      0.85        13

    accuracy                           0.83        24
   macro avg       0.83      0.83      0.83        24
weighted avg       0.83      0.83      0.83        24

[0.0564418 0.9435582]
[0.09562297 0.90437703]


In [6]:
import gradio as gr

def heart_fn(age, cholesterol, blood_pressure, smoker):
    p = predict_proba("heart", [age, cholesterol, blood_pressure, smoker])
    labels = CONFIG["heart"]["labels"]
    return {labels[0]: float(p[0]), labels[1]: float(p[1])}

def loan_fn(income, credit_score, loan_amount, employed):
    p = predict_proba("loan", [income, credit_score, loan_amount, employed])
    labels = CONFIG["loan"]["labels"]
    return {labels[0]: float(p[0]), labels[1]: float(p[1])}

heart_ui = gr.Interface(
    fn=heart_fn,
    inputs=[gr.Number(label="Age", value=50),
            gr.Number(label="Cholesterol", value=220),
            gr.Number(label="Blood Pressure", value=130),
            gr.Radio([0, 1], label="Smoker (0 = No, 1 = Yes)", value=0)],
    outputs=gr.Label(label="Prediction"),
    title="Heart Disease Risk Predictor")

loan_ui = gr.Interface(
    fn=loan_fn,
    inputs=[gr.Number(label="Income (thousands)", value=60),
            gr.Number(label="Credit Score", value=650),
            gr.Number(label="Loan Amount (thousands)", value=25),
            gr.Radio([0, 1], label="Employed (0 = No, 1 = Yes)", value=1)],
    outputs=gr.Label(label="Prediction"),
    title="Loan Approval Predictor")

demo = gr.TabbedInterface([heart_ui, loan_ui], ["Heart Disease", "Loan Approval"])
demo.launch()   # opens inside the notebook; use share=True for a public link

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
from flask import Flask, request, render_template_string
from werkzeug.serving import make_server
import threading

app = Flask(__name__)

PAGE = """
<!doctype html>
<title>{{ title }}</title>
<body style="font-family:Arial; max-width:480px; margin:40px auto;">
  <h2>{{ title }}</h2>
  <p><a href="/heart">Heart Disease</a> | <a href="/loan">Loan Approval</a></p>
  <form method="post">
    {% for f in features %}
      <label>{{ f }}</label><br>
      <input type="number" step="any" name="{{ f }}" value="{{ values.get(f, '') }}" required
             style="width:100%; padding:6px; margin-bottom:10px;"><br>
    {% endfor %}
    <button type="submit" style="padding:8px 16px;">Predict</button>
  </form>
  {% if result %}
    <h3>Result: {{ result }}</h3>
    <p>Confidence: {{ confidence }}%</p>
  {% endif %}
</body>
"""

def handle(name, title):
    result = confidence = None
    values = {}
    if request.method == "POST":
        values = {f: float(request.form[f]) for f in FEATURES[name]}
        p = predict_proba(name, [values[f] for f in FEATURES[name]])
        cls = int(p.argmax())
        result = CONFIG[name]["labels"][cls]
        confidence = round(float(p[cls]) * 100, 2)
    return render_template_string(PAGE, title=title, features=FEATURES[name],
                                  values=values, result=result, confidence=confidence)

@app.route("/", methods=["GET", "POST"])
@app.route("/heart", methods=["GET", "POST"])
def heart_page():
    return handle("heart", "Heart Disease Risk Predictor")

@app.route("/loan", methods=["GET", "POST"])
def loan_page():
    return handle("loan", "Loan Approval Predictor")

# Run Flask in a background thread so the notebook stays usable
server = make_server("127.0.0.1", 5001, app)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print("Flask running at http://127.0.0.1:5001/heart and http://127.0.0.1:5001/loan")

Flask running at http://127.0.0.1:5001/heart and http://127.0.0.1:5001/loan


127.0.0.1 - - [14/Oct/2026 12:12:51] "GET /heart HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:12:51] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [14/Oct/2026 12:14:16] "GET /loan HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:14:19] "GET /heart HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:15:18] "POST /heart HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:15:27] "GET /loan HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:15:29] "GET /heart HTTP/1.1" 200 -


Created dataset file at: .gradio\flagged\dataset1.csv


127.0.0.1 - - [14/Oct/2026 12:24:42] "POST /heart HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:25:47] "GET /loan HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 12:26:08] "POST /loan HTTP/1.1" 200 -
